# 感知机（Perceptron）

感知机是最早的机器学习算法之一（Rosenblatt，1957），也是神经网络的基础单元。

**核心思想**：模拟生物神经元——对输入加权求和，超过阈值则激活输出 1，否则输出 -1。

感知机是一个**二分类线性模型**，只能处理线性可分数据。

## 1. 模型定义

感知机的决策函数：

$$f(x) = \text{sign}(w \cdot x + b) = \begin{cases} +1 & w \cdot x + b > 0 \\ -1 & w \cdot x + b \leq 0 \end{cases}$$

- $w \in \mathbb{R}^n$：权重向量（法向量），决定超平面方向
- $b \in \mathbb{R}$：偏置，决定超平面位置
- 决策边界 $w \cdot x + b = 0$ 是一个超平面，将特征空间分为两半

## 2. 损失函数

不能用误分类数量作为损失（不连续、无法求导）。

感知机损失 = **所有误分类点到超平面的距离之和**（去掉常数分母 $\|w\|$）：

$$L(w, b) = -\sum_{x_i \in M} y_i (w \cdot x_i + b)$$

其中 $M$ 是所有被误分类的样本集合。

- 误分类时 $y_i(w \cdot x_i + b) < 0$，取负号后损失为正
- 正确分类时不贡献损失
- 损失越大说明误分类点离超平面越远

## 3. 随机梯度下降更新规则

每次随机选一个误分类点 $(x_i, y_i)$，对参数做梯度更新：

$$\frac{\partial L}{\partial w} = -y_i x_i, \quad \frac{\partial L}{\partial b} = -y_i$$

$$w \leftarrow w + \eta y_i x_i$$
$$b \leftarrow b + \eta y_i$$

- $\eta$：学习率，控制更新步长
- 每次只用**一个误分类点**更新（随机梯度下降）
- 重复直到没有误分类点（收敛）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 生成线性可分的二分类数据，标签为 +1 / -1
np.random.seed(42)
X_pos = np.random.randn(30, 2) + np.array([2, 2])
X_neg = np.random.randn(30, 2) + np.array([-2, -2])
X = np.vstack([X_pos, X_neg])
y = np.array([1] * 30 + [-1] * 30)

def sign(z):
    return 1 if z > 0 else -1

def perceptron_fit(X, y, eta=0.1, max_iter=1000):
    w = np.zeros(X.shape[1])
    b = 0.0
    history = []      # 记录每次更新的 (w, b)
    for _ in range(max_iter):
        misclassified = False
        for xi, yi in zip(X, y):
            if sign(w @ xi + b) != yi:       # 找到误分类点
                w += eta * yi * xi
                b += eta * yi
                history.append((w.copy(), b))
                misclassified = True
        if not misclassified:
            break
    return w, b, history

w, b, history = perceptron_fit(X, y, eta=0.1)
print(f"收敛，共更新 {len(history)} 次")
print(f"最终权重 w = {w.round(4)}, b = {b:.4f}")

## 4. 决策边界可视化

In [ ]:
def plot_boundary(ax, X, y, w, b, title):
    ax.scatter(X[y==1,  0], X[y==1,  1], c='steelblue', edgecolors='k', s=40, label='+1')
    ax.scatter(X[y==-1, 0], X[y==-1, 1], c='tomato',    edgecolors='k', s=40, label='-1')
    x1 = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200)
    if abs(w[1]) > 1e-6:
        x2 = -(w[0] * x1 + b) / w[1]
        ax.plot(x1, x2, 'k-', linewidth=2, label='决策边界')
    ax.set_title(title)
    ax.legend(fontsize=8)

# 展示初始、中间、最终三个状态的决策边界
checkpoints = [0, len(history) // 2, len(history) - 1]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, idx in zip(axes, checkpoints):
    wi, bi = history[idx]
    plot_boundary(ax, X, y, wi, bi, f'第 {idx+1} 次更新后')
plt.suptitle('感知机决策边界收敛过程', fontsize=13)
plt.tight_layout()
plt.show()

## 5. 收敛性定理（Novikoff 定理）

若数据**线性可分**，感知机算法**一定在有限步内收敛**。

更新次数上界：

$$\text{更新次数} \leq \left(\frac{R}{\gamma}\right)^2$$

- $R = \max_i \|x_i\|$：样本到原点的最大距离
- $\gamma$：间隔（数据到最优超平面的最小距离）
- 间隔 $\gamma$ 越大，数据越容易分，收敛越快

> 若数据**线性不可分**，感知机永远不会收敛（无限循环）。这正是 SVM 和神经网络被发展出来的原因。

In [ ]:
# 验证：线性不可分（XOR）时感知机不收敛
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([-1, 1, 1, -1])

_, _, hist_xor = perceptron_fit(X_xor, y_xor, eta=0.1, max_iter=200)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# 可分数据：更新次数随迭代趋于稳定
ax1.plot(range(1, len(history)+1), range(1, len(history)+1), 'steelblue')
ax1.set_title(f'线性可分数据（共更新 {len(history)} 次，收敛）')
ax1.set_xlabel('更新次数'); ax1.set_ylabel('累计更新次数')

# XOR：一直在更新，不收敛
ax2.plot(range(1, len(hist_xor)+1), range(1, len(hist_xor)+1), 'tomato')
ax2.set_title(f'XOR（线性不可分，{len(hist_xor)} 步仍未收敛）')
ax2.set_xlabel('更新次数'); ax2.set_ylabel('累计更新次数')

plt.tight_layout()
plt.show()

## 6. 感知机的对偶形式

将 $w$ 用训练样本线性表示：

$$w = \sum_{i=1}^{n} \alpha_i y_i x_i, \quad b = \sum_{i=1}^{n} \alpha_i y_i$$

其中 $\alpha_i$ 表示第 $i$ 个样本**被用来更新参数的次数**。

判断条件变为：若 $y_i \left(\sum_j \alpha_j y_j x_j \cdot x_i + b\right) \leq 0$ 则更新。

对偶形式的核心：内积 $x_j \cdot x_i$ 可预先用 **Gram 矩阵**计算好，避免重复运算——这也是 SVM 核函数思想的雏形。

In [ ]:
def perceptron_dual(X, y, eta=0.1, max_iter=1000):
    n = len(X)
    alpha = np.zeros(n)
    b = 0.0
    G = X @ X.T   # Gram 矩阵，预先计算所有内积
    for _ in range(max_iter):
        updated = False
        for i in range(n):
            val = y[i] * (np.sum(alpha * y * G[:, i]) + b)
            if val <= 0:            # 误分类
                alpha[i] += eta
                b += eta * y[i]
                updated = True
        if not updated:
            break
    w = X.T @ (alpha * y)
    return w, b, alpha

w_dual, b_dual, alpha = perceptron_dual(X, y)
print("对偶形式结果：")
print(f"  w = {w_dual.round(4)}, b = {b_dual:.4f}")
print(f"  非零 alpha（支撑样本）数量: {np.sum(alpha > 0)}")

# 验证与原始形式结果一致
preds = np.sign(X @ w_dual + b_dual)
print(f"  训练准确率: {np.mean(preds == y):.2%}")

## 7. sklearn 实现感知机

In [ ]:
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

# 鸢尾花数据集（取前两类做二分类）
iris = load_iris()
mask = iris.target < 2
X_ir, y_ir = iris.data[mask], iris.target[mask] * 2 - 1  # 标签转为 +1/-1

X_tr, X_te, y_tr, y_te = train_test_split(X_ir, y_ir, test_size=0.2, random_state=42)
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s  = sc.transform(X_te)

ppn = Perceptron(eta0=0.1, max_iter=1000, random_state=42)
ppn.fit(X_tr_s, y_tr)

print(f"训练集准确率: {accuracy_score(y_tr, ppn.predict(X_tr_s)):.4f}")
print(f"测试集准确率: {accuracy_score(y_te, ppn.predict(X_te_s)):.4f}")
print(f"迭代次数: {ppn.n_iter_}")
print(f"权重向量 w: {ppn.coef_.round(4)}")
print(f"偏置 b: {ppn.intercept_}")

## 总结

| 知识点 | 核心内容 |
|--------|----------|
| 模型 | $f(x) = \text{sign}(w \cdot x + b)$，线性超平面分类 |
| 损失函数 | 误分类点到超平面的距离之和，$L = -\sum_{M} y_i(w \cdot x_i + b)$ |
| 更新规则 | $w \leftarrow w + \eta y_i x_i$，$b \leftarrow b + \eta y_i$ |
| 收敛性 | 线性可分时有限步收敛；线性不可分时不收敛 |
| 对偶形式 | $w = \sum \alpha_i y_i x_i$，引出 Gram 矩阵与核方法思想 |

**感知机 → SVM 的进化**：
- 感知机找到**任意一个**可分超平面（不唯一）
- SVM 在感知机基础上增加「最大间隔」约束，找到**唯一最优**超平面
- 感知机的对偶形式与 Gram 矩阵 → SVM 的核技巧